In [2]:
import json

import numpy as np
import pandas as pd
import pyod
import sklearn
from ucimlrepo import fetch_ucirepo
from pyod.models.hbos import HBOS
from sklearn.metrics import roc_auc_score

In [3]:
PAPER_AUC = 0.9910

# Fetch Breast Cancer Wisconsin (Diagnostic), UCI dataset ID 17.
dataset = fetch_ucirepo(id=17)
X = dataset.data.features.apply(pd.to_numeric, errors="coerce")
diagnosis = (
    dataset.data.targets.iloc[:, 0]
    .astype(str)
    .str.strip()
    .str.upper()
)

# Preserve complete rows and their original order.
complete = X.notna().all(axis=1) & diagnosis.notna()
X = X.loc[complete]
diagnosis = diagnosis.loc[complete]

# Amer (2011), Section 5.2.1: retain all benign cases and
# the first 10 malignant cases.
first_10_malignant = diagnosis[diagnosis == "M"].index[:10]
keep = diagnosis.eq("B") | diagnosis.index.isin(first_10_malignant)

X = X.loc[keep].to_numpy()
y = diagnosis.loc[keep].eq("M").astype(int).to_numpy()

# 367 rows after preprocessing; round(sqrt(367)) = 19 bins.
# Labels are not used to fit the unsupervised detector.
model = HBOS(
    n_bins=19,
    alpha=0.1,
    tol=0.5,
    contamination=0.1,
)
model.fit(X)

auc = roc_auc_score(y, model.decision_scores_)

result = {
    "dataset": "Breast Cancer Wisconsin (Diagnostic), preprocessed per Amer (2011)",
    "rows": int(len(X)),
    "features": int(X.shape[1]),
    "benign_records": int((y == 0).sum()),
    "malignant_anomalies": int(y.sum()),
    "parameters": {
        "n_bins": 19,
        "alpha": 0.1,
        "tol": 0.5,
        "contamination": 0.1,
    },
    "paper_auc": PAPER_AUC,
    "reproduced_auc": float(auc),
    "absolute_auc_difference": float(abs(auc - PAPER_AUC)),
    "pyod_version": pyod.__version__,
    "scikit_learn_version": sklearn.__version__,
}

print(json.dumps(result, indent=2))

{
  "dataset": "Breast Cancer Wisconsin (Diagnostic), preprocessed per Amer (2011)",
  "rows": 367,
  "features": 30,
  "benign_records": 357,
  "malignant_anomalies": 10,
  "parameters": {
    "n_bins": 19,
    "alpha": 0.1,
    "tol": 0.5,
    "contamination": 0.1
  },
  "paper_auc": 0.991,
  "reproduced_auc": 0.9521008403361344,
  "absolute_auc_difference": 0.03889915966386559,
  "pyod_version": "3.6.4",
  "scikit_learn_version": "1.9.0"
}
